parseamos todos los patterns y comprobamos que encontramos todos

In [1]:
import pandas as pd
from pathlib import Path

PATTERNS_PATH = Path("data/HI-Medium_Patterns.txt")

def get_base_pattern(header: str) -> str:
    """
    header viene de la parte después de 'BEGIN LAUNDERING ATTEMPT -'
    y puede ser, por ejemplo:
      'STACK'
      'CYCLE Max 12 hops'
      'FAN-IN Max 9-degree Fan-In'
      'GATHER-SCATTER Max 8-degree Fan-In'
      'SCATTER-GATHER'
      'RANDOM Max 3 hops'
    """
    h = header.upper().strip()

    # Ojo al orden: comprobamos los nombres completos antes
    if "GATHER-SCATTER" in h and "SCATTER-GATHER" not in h:
        return "GATHER-SCATTER"
    if "SCATTER-GATHER" in h and "GATHER-SCATTER" not in h:
        return "SCATTER-GATHER"

    if "STACK" in h:
        return "STACK"
    if "FAN-IN" in h or "FAN IN" in h:
        return "FAN-IN"
    if "FAN-OUT" in h or "FAN OUT" in h:
        return "FAN-OUT"
    if "BIPARTITE" in h:
        return "BIPARTITE"
    if "CYCLE" in h:
        return "CYCLE"
    if "RANDOM" in h:
        return "RANDOM"

    return "UNKNOWN"


def parse_patterns_with_base(patterns_path: Path) -> pd.DataFrame:
    """
    Lee HI-Medium_Patterns.txt y devuelve un DataFrame con:
      timestamp, id_origen, cuenta_origen, id_destino, cuenta_destino,
      monto, moneda_out, monto_in, moneda_in, metodo, flag, pattern_base
    """
    rows = []
    current_pattern_base = None

    with open(patterns_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue

            # Detectar inicio de patrón
            if line.startswith("BEGIN LAUNDERING ATTEMPT -"):
                header = line.split("BEGIN LAUNDERING ATTEMPT -", 1)[1].strip()
                current_pattern_base = get_base_pattern(header)
                continue

            # Detectar fin de patrón
            if line.startswith("END LAUNDERING ATTEMPT -"):
                current_pattern_base = None
                continue

            # Si estamos dentro de un patrón y la línea parece una transacción
            if current_pattern_base is not None and "," in line:
                parts = [p.strip() for p in line.split(",")]
                # Formato esperado: 11 columnas (como ya tenías):
                # timestamp, id_origen, cuenta_origen, id_destino, cuenta_destino,
                # monto, moneda_out, monto_in, moneda_in, metodo, flag
                if len(parts) < 11:
                    continue  # línea rara; la ignoramos

                timestamp = parts[0]
                id_origen = parts[1]
                cuenta_origen = parts[2]
                id_destino = parts[3]
                cuenta_destino = parts[4]
                monto = parts[5]
                moneda_out = parts[6]
                monto_in = parts[7]
                moneda_in = parts[8]
                metodo = parts[9]
                flag = parts[10]

                rows.append(
                    {
                        "timestamp": timestamp,
                        "id_origen": id_origen,
                        "cuenta_origen": cuenta_origen,
                        "id_destino": id_destino,
                        "cuenta_destino": cuenta_destino,
                        "monto": float(monto),
                        "moneda_out": moneda_out,
                        "monto_in": float(monto_in),
                        "moneda_in": moneda_in,
                        "metodo": metodo,
                        "flag": int(flag),
                        "pattern_base": current_pattern_base,
                    }
                )

    return pd.DataFrame(rows)

patterns_df = parse_patterns_with_base(PATTERNS_PATH)

print(patterns_df.head())
print("\nClases generales detectadas:")
print(patterns_df["pattern_base"].value_counts())


          timestamp id_origen cuenta_origen id_destino cuenta_destino  \
0  2022/09/01 05:14     00952     8139F54E0    0111632      8062C56E0   
1  2022/09/03 13:09   0111632     8062C56E0     008456      81363F620   
2  2022/09/01 07:40   0118693     823D5EB90     013729      801CF2E60   
3  2022/09/01 14:19    013729     801CF2E60    0123621      81A7090F0   
4  2022/09/02 12:40   0024750     81363F410    0213834      808757B00   

      monto moneda_out  monto_in  moneda_in metodo  flag pattern_base  
0   5331.44  US Dollar   5331.44  US Dollar    ACH     1        STACK  
1   5602.59  US Dollar   5602.59  US Dollar    ACH     1        STACK  
2   1400.54  US Dollar   1400.54  US Dollar    ACH     1        STACK  
3   1467.94  US Dollar   1467.94  US Dollar    ACH     1        STACK  
4  16898.29  US Dollar  16898.29  US Dollar    ACH     1        STACK  

Clases generales detectadas:
pattern_base
GATHER-SCATTER    4289
SCATTER-GATHER    3988
STACK             3986
FAN-IN           

In [2]:
import pandas as pd
from pathlib import Path

# =====================================================
# 1. CARGAR CSV PRINCIPAL
# =====================================================
DATA_PATH = Path("data/HI-Medium_Trans.csv")
df = pd.read_csv(DATA_PATH)

print("Transacciones cargadas:", df.shape)
print(df.head(3))

# patterns_df ya lo tienes creado con parse_patterns_with_base(...)
print("Patterns parseados:", patterns_df.shape)
print(patterns_df["pattern_base"].value_counts())

# =====================================================
# 2. PREPARAR patterns_df PARA EL MERGE
# =====================================================
# Renombrar columnas para que coincidan con las del CSV
patterns_merge = patterns_df.rename(columns={
    "id_origen": "From Bank",
    "cuenta_origen": "Account",
    "id_destino": "To Bank",
    "cuenta_destino": "Account.1",
    "monto": "Amount Received",
    "metodo": "Payment Format",
})

# Nos quedamos solo con las columnas necesarias para unir + pattern_base
patterns_merge = patterns_merge[
    ["From Bank", "Account", "To Bank", "Account.1",
     "Amount Received", "Payment Format", "pattern_base"]
].copy()

# Asegurar tipos compatibles con df
# En el CSV, From Bank y To Bank suelen ser enteros; aquí convertimos
patterns_merge["From Bank"] = patterns_merge["From Bank"].astype(int)
patterns_merge["To Bank"]   = patterns_merge["To Bank"].astype(int)
# Amount Received ya es float en patterns_df; en df también debería ser float

print("\npatterns_merge ejemplo:")
print(patterns_merge.head())

# =====================================================
# 3. MERGE CON EL CSV PRINCIPAL
# =====================================================
merge_keys = ["From Bank", "Account", "To Bank", "Account.1",
              "Amount Received", "Payment Format"]

# Comprobación rápida de que las claves existen en ambos
print("\nColumnas df:", df.columns.tolist())
print("Columnas patterns_merge:", patterns_merge.columns.tolist())

missing_in_df = [c for c in merge_keys if c not in df.columns]
missing_in_patterns = [c for c in merge_keys if c not in patterns_merge.columns]
print("Faltan en df:", missing_in_df)
print("Faltan en patterns_merge:", missing_in_patterns)

# Merge left: todas las transacciones del CSV, patrón solo si coincide
df_merged = df.merge(
    patterns_merge,
    on=merge_keys,
    how="left"
)

print("\nResultado merge, primeras filas:")
print(df_merged[["Timestamp", "From Bank", "Account", "pattern_base"]].head(10))

print("\nDistribución pattern_base (incluyendo NaN):")
print(df_merged["pattern_base"].value_counts(dropna=False))

# =====================================================
# 4. CREAR TARGET MULTI-CLASE 0–8
# =====================================================
pattern_to_id = {
    "STACK": 1,
    "CYCLE": 2,
    "FAN-IN": 3,
    "FAN-OUT": 4,
    "GATHER-SCATTER": 5,
    "SCATTER-GATHER": 6,
    "BIPARTITE": 7,
    "RANDOM": 8,
}

df_merged["target_multi"] = df_merged["pattern_base"].map(pattern_to_id)
df_merged["target_multi"] = df_merged["target_multi"].fillna(0).astype(int)  # 0 = normal / no tipificado

print("\nDistribución target_multi (0–8):")
print(df_merged["target_multi"].value_counts().sort_index())


Transacciones cargadas: (31898238, 11)
          Timestamp  From Bank    Account  To Bank  Account.1  \
0  2022/09/01 00:17         20  800104D70       20  800104D70   
1  2022/09/01 00:02       3196  800107150     3196  800107150   
2  2022/09/01 00:17       1208  80010E430     1208  80010E430   

   Amount Received Receiving Currency  Amount Paid Payment Currency  \
0          6794.63          US Dollar      6794.63        US Dollar   
1          7739.29          US Dollar      7739.29        US Dollar   
2          1880.23          US Dollar      1880.23        US Dollar   

  Payment Format  Is Laundering  
0   Reinvestment              0  
1   Reinvestment              0  
2   Reinvestment              0  
Patterns parseados: (22743, 12)
pattern_base
GATHER-SCATTER    4289
SCATTER-GATHER    3988
STACK             3986
FAN-IN            2315
CYCLE             2235
BIPARTITE         2135
FAN-OUT           2128
RANDOM            1667
Name: count, dtype: int64

patterns_merge ejemplo:

In [3]:
df_merged[df_merged["target_multi"] == 2].head()

,Timestamp,From Bank,Account,To Bank,Account.1,Amount Received,Receiving Currency,Amount Paid,Payment Currency,Payment Format,Is Laundering,pattern_base,target_multi
232646,2022/09/01 00:19,134266,814167590,36925,810E343A0,132713.46,Yuan,132713.46,Yuan,ACH,1,CYCLE,2
232648,2022/09/01 19:35,36925,810E343A0,119211,814AB4F60,18264.20,US Dollar,18264.20,US Dollar,ACH,1,CYCLE,2
232650,2022/09/02 02:58,119211,814AB4F60,132965,81B88A230,14567.69,Euro,14567.69,Euro,ACH,1,CYCLE,2
232653,2022/09/02 18:02,132965,81B88A230,137089,810C71940,114329.26,Yuan,114329.26,Yuan,ACH,1,CYCLE,2
232656,2022/09/03 07:16,137089,810C71940,216618,81D5302D0,14567.69,Euro,14567.69,Euro,ACH,1,CYCLE,2


In [4]:
# =====================================================
# 5. GUARDAR DATASET FINAL
# =====================================================
df_merged.to_csv("data/ibm_aml_multiclass_clases.csv", index=False)
print("\nDataset multiclase guardado en data/ibm_aml_multiclass_clases.csv")



Dataset multiclase guardado en data/ibm_aml_multiclass_clases.csv
